# 04 -- Mineria de Procesos y Deteccion de Cuellos de Botella

**Objetivo**: Construir el flujo de procesos (Sankey diagram data), detectar cuellos de botella operacionales, realizar analisis Pareto y generar una matriz de prioridades para recomendaciones accionables.

**Entrada**: `data/processed/nyc311_enriched.parquet`

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_parquet(Path('../data/processed/nyc311_enriched.parquet'))
print(f'Datos cargados: {len(df):,} filas')

## 1. Construccion del Flujo de Procesos (Sankey)

Modelamos el flujo: **Tipo de Queja** -> **Agencia** -> **Etapa del Proceso** -> **Categoria de Resolucion**

Esto simula una vista de mineria de procesos, mostrando como fluyen las solicitudes a traves del sistema.

In [ ]:
# Filtrar datos completos para el Sankey
sankey_df = df.dropna(subset=['complaint_category', 'agency_name', 'process_stage', 'response_category'])
print(f'Filas con datos completos para Sankey: {len(sankey_df):,}')

# Top 10 categorias y agencias
top_complaints = sankey_df['complaint_category'].value_counts().head(10).index.tolist()
top_agencies = sankey_df['agency_name'].value_counts().head(10).index.tolist()

sankey_filtered = sankey_df[
    sankey_df['complaint_category'].isin(top_complaints) &
    sankey_df['agency_name'].isin(top_agencies)
]
print(f'Filas despues de filtrar top 10: {len(sankey_filtered):,}')

In [ ]:
# Construir nodos y enlaces
stages = sankey_filtered['process_stage'].unique().tolist()
response_cats = sankey_filtered['response_category'].unique().tolist()

all_nodes = top_complaints + top_agencies + stages + response_cats
node_indices = {name: i for i, name in enumerate(all_nodes)}

# Enlaces: complaint -> agency
links_1 = sankey_filtered.groupby(['complaint_category', 'agency_name']).size().reset_index(name='value')
links_1.columns = ['source', 'target', 'value']

# Enlaces: agency -> process_stage  
links_2 = sankey_filtered.groupby(['agency_name', 'process_stage']).size().reset_index(name='value')
links_2.columns = ['source', 'target', 'value']

# Enlaces: process_stage -> response_category
links_3 = sankey_filtered.groupby(['process_stage', 'response_category']).size().reset_index(name='value')
links_3.columns = ['source', 'target', 'value']

all_links = pd.concat([links_1, links_2, links_3], ignore_index=True)
all_links['source_idx'] = all_links['source'].map(node_indices)
all_links['target_idx'] = all_links['target'].map(node_indices)
all_links = all_links.dropna(subset=['source_idx', 'target_idx'])

print(f'Nodos: {len(all_nodes)}, Enlaces: {len(all_links)}')

In [ ]:
# Visualizacion Sankey
node_colors = (
    ['#3B82F6'] * len(top_complaints) +
    ['#06B6D4'] * len(top_agencies) +
    ['#10B981'] * len(stages) +
    ['#F59E0B'] * len(response_cats)
)

fig = go.Figure(go.Sankey(
    node=dict(
        pad=15, thickness=20,
        line=dict(color='#374151', width=0.5),
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=all_links['source_idx'].astype(int).tolist(),
        target=all_links['target_idx'].astype(int).tolist(),
        value=all_links['value'].tolist(),
        color='rgba(59, 130, 246, 0.2)'
    )
))

fig.update_layout(
    title='Flujo de Procesos: Tipo de Queja -> Agencia -> Etapa -> Resolucion',
    height=600, template='plotly_dark',
    font=dict(size=10)
)
fig.show()

## 2. Deteccion de Cuellos de Botella

Identificamos las combinaciones agencia + tipo de queja con los tiempos de resolucion mas altos -- estos son los cuellos de botella operacionales.

In [ ]:
bottleneck = df.dropna(subset=['resolution_days']).groupby(['agency_name', 'complaint_category']).agg(
    promedio_dias=('resolution_days', 'mean'),
    mediana_dias=('resolution_days', 'median'),
    solicitudes=('unique_key', 'count'),
    pct_incumple=('is_overdue', 'mean')
).reset_index()

bottleneck = bottleneck[bottleneck['solicitudes'] >= 50].sort_values('promedio_dias', ascending=False)

print('Top 10 Cuellos de Botella (mayor tiempo promedio de resolucion):\n')
bottleneck.head(10)

In [ ]:
# Visualizacion: scatter de cuellos de botella
fig = px.scatter(
    bottleneck.head(30),
    x='solicitudes', y='promedio_dias',
    size='solicitudes', color='pct_incumple',
    hover_data=['agency_name', 'complaint_category'],
    title='Cuellos de Botella: Volumen vs Tiempo de Resolucion',
    labels={'solicitudes': 'Volumen de Solicitudes', 'promedio_dias': 'Dias Promedio',
            'pct_incumple': 'Tasa Incumplimiento'},
    color_continuous_scale=[[0, '#10B981'], [0.5, '#F59E0B'], [1, '#EF4444']]
)
fig.update_layout(height=500, template='plotly_dark')
fig.show()

## 3. Analisis Pareto (80/20)

In [ ]:
pareto = df['complaint_category'].value_counts().reset_index()
pareto.columns = ['tipo_queja', 'solicitudes']
pareto['pct'] = pareto['solicitudes'] / pareto['solicitudes'].sum() * 100
pareto['pct_acumulado'] = pareto['pct'].cumsum()

# Cuantos tipos generan el 80%?
tipos_80 = len(pareto[pareto['pct_acumulado'] <= 80]) + 1
total_tipos = len(pareto)
print(f'{tipos_80} de {total_tipos} tipos de queja ({tipos_80/total_tipos:.0%}) generan el 80% del volumen')

In [ ]:
# Grafico Pareto dual-eje
fig = make_subplots(specs=[[{'secondary_y': True}]])

colors = ['#3B82F6' if row['pct_acumulado'] <= 80 else '#374151' 
          for _, row in pareto.iterrows()]

fig.add_trace(
    go.Bar(x=pareto['tipo_queja'], y=pareto['solicitudes'], 
           name='Solicitudes', marker_color=colors),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=pareto['tipo_queja'], y=pareto['pct_acumulado'],
               name='% Acumulado', line=dict(color='#06B6D4', width=2),
               mode='lines+markers'),
    secondary_y=True
)

fig.add_hline(y=80, line_dash='dash', line_color='#F59E0B', secondary_y=True,
              annotation_text='80%')

fig.update_layout(
    title='Analisis Pareto: Tipos de Queja',
    height=500, template='plotly_dark',
    xaxis_tickangle=45
)
fig.update_yaxes(title_text='Solicitudes', secondary_y=False)
fig.update_yaxes(title_text='% Acumulado', secondary_y=True)
fig.show()

## 4. Matriz de Prioridades

Combinamos volumen y tiempo de resolucion para priorizar donde enfocarse:
- **Cuadrante superior derecho**: Alto volumen + alta resolucion = prioridad critica
- **Cuadrante inferior derecho**: Alto volumen + baja resolucion = bien gestionado
- **Cuadrante superior izquierdo**: Bajo volumen + alta resolucion = ineficiencia puntual
- **Cuadrante inferior izquierdo**: Bajo volumen + baja resolucion = sin accion

In [ ]:
priority = df.dropna(subset=['resolution_days']).groupby('complaint_category').agg(
    volumen=('unique_key', 'count'),
    promedio_dias=('resolution_days', 'mean'),
    tasa_cumplimiento=('sla_status', lambda x: (x == 'Cumple').mean())
).reset_index()
priority = priority[priority['volumen'] >= 100]

vol_median = priority['volumen'].median()
days_median = priority['promedio_dias'].median()

fig = px.scatter(
    priority,
    x='volumen', y='promedio_dias',
    size='volumen', color='tasa_cumplimiento',
    text='complaint_category',
    title='Matriz de Prioridades: Volumen vs Tiempo de Resolucion',
    labels={'volumen': 'Volumen de Solicitudes', 'promedio_dias': 'Dias Promedio de Resolucion',
            'tasa_cumplimiento': 'Cumplimiento SLA'},
    color_continuous_scale=[[0, '#EF4444'], [0.5, '#F59E0B'], [1, '#10B981']]
)
fig.add_vline(x=vol_median, line_dash='dot', line_color='#9CA3AF')
fig.add_hline(y=days_median, line_dash='dot', line_color='#9CA3AF')
fig.update_traces(textposition='top center', textfont_size=8)
fig.update_layout(height=600, template='plotly_dark')
fig.show()

## 5. Resumen Ejecutivo y Recomendaciones

### Hallazgos Principales

1. **Flujo de procesos**: La mayoria de solicitudes fluyen de un numero reducido de tipos de queja hacia agencias especificas, creando concentracion de carga
2. **Cuellos de botella**: Combinaciones especificas de agencia-queja presentan tiempos de resolucion anormalmente altos
3. **Regla 80/20**: Un pequeno porcentaje de tipos de queja genera la gran mayoria del volumen
4. **Prioridades claras**: La matriz identifica quejas de alto volumen Y alto tiempo de resolucion como areas de intervencion inmediata

### Recomendaciones

1. **Redistribuir carga**: Las agencias con cuellos de botella necesitan recursos adicionales o reasignacion de tipos de queja
2. **Automatizar resoluciones**: Los tipos de queja mas frecuentes con resoluciones rapidas son candidatos para autoservicio/chatbot
3. **SLA diferenciados**: Establecer SLAs especificos por tipo de queja en lugar de un umbral unico
4. **Monitoreo continuo**: Implementar alertas tempranas cuando una agencia supere su tiempo mediano de resolucion

---

**Fin del pipeline analitico.** Todos los datos procesados estan disponibles en `data/processed/nyc311_enriched.parquet` para alimentar el dashboard interactivo.